In [93]:
# pip install yfinance pandas numpy matplotlib ta


In [94]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:.2f}".format)


In [95]:
TICKERS = [
    # Broad Market
    "SPY","QQQ","DIA","IWM",

    # Mega Cap Tech
    "AAPL","MSFT","NVDA","GOOGL","AMZN",
    "META","TSLA","AVGO","ORCL","AMD",

    # Financials
    "JPM","BAC","GS","MS","BLK",

    # Healthcare
    "LLY","UNH","JNJ","ABBV","MRK",

    # Industrials
    "CAT","GE","RTX","DE","HON",

    # Consumer
    "COST","WMT","HD","MCD","SBUX",

    # Energy
    "XOM","CVX","COP","SLB",

    # Semiconductors
    "MU","QCOM","TXN","INTC","ASML",

    # ETFs
    "VTI","VOO","SCHD","VXUS","XLK","XLF","XLE"
]
START_DATE = "2018-01-01"
MA = 30


In [96]:
def load_weekly_data(ticker):
    df = yf.Ticker(ticker).history(
        start=START_DATE,
        interval="1d",
        auto_adjust=True
    )

    # Keep only Close and force Series
    close = df["Close"].astype(float)

    # Weekly resample
    close = close.resample("W-FRI").last().dropna()

    return pd.DataFrame({"Close": close})


In [97]:
def compute_weekly_signals(df):
    df = df.copy()

    close = pd.Series(df["Close"].values, index=df.index)

    # Defensive checks
    assert close.ndim == 1, f"Close is not 1D: shape={close.shape}"

    df["ma40"] = close.rolling(MA).mean()
    df["rsi"] = ta.momentum.RSIIndicator(close, window=14).rsi()

    df["signal"] = "HOLD"
    df.loc[(close > df["ma40"]) & (df["rsi"] < 80), "signal"] = "BUY"
    df.loc[close < df["ma40"], "signal"] = "SELL"

    return df


In [98]:
def backtest(df):
    df = df.copy()

    df["position"] = 0
    df.loc[df["signal"] == "BUY", "position"] = 1
    df.loc[df["signal"] == "SELL", "position"] = 0
    df["position"] = df["position"].ffill().fillna(0)

    df["weekly_return"] = df["Close"].pct_change()
    df["strategy_return"] = df["position"].shift(1) * df["weekly_return"]

    df["cum_market"] = (1 + df["weekly_return"]).cumprod()
    df["cum_strategy"] = (1 + df["strategy_return"]).cumprod()

    return df


In [99]:
results = {}
summaries = []

for ticker in TICKERS:
    df = load_weekly_data(ticker)
    df = compute_weekly_signals(df)
    df = backtest(df)

    results[ticker] = df

    total_return = df["cum_strategy"].iloc[-1] - 1
    market_return = df["cum_market"].iloc[-1] - 1

    summaries.append({
        "Ticker": ticker,
        "Strategy Return %": total_return * 100,
        "Market Return %": market_return * 100,
        "Outperformed": total_return > market_return
    })

summary_df = pd.DataFrame(summaries)
summary_df


,Ticker,Strategy Return %,Market Return %,Outperformed
0,SPY,139.63,205.96,False
1,QQQ,230.91,366.67,False
2,DIA,36.64,139.97,False
3,IWM,47.84,112.03,False
4,AAPL,91.47,627.33,False
5,MSFT,159.02,361.86,False
6,NVDA,1014.43,3679.29,False
7,GOOGL,166.35,527.35,False
8,AMZN,59.91,280.02,False
9,META,363.28,204.01,True


In [100]:
weekly_signals = []

for ticker, df in results.items():
    latest = df.iloc[-1]
    weekly_signals.append({
        "Ticker": ticker,
        "Signal": latest["signal"],
        "Price": latest["Close"],
        "RSI": latest["rsi"]
    })

signals_df = pd.DataFrame(weekly_signals)
signals_df.sort_values("Signal")


,Ticker,Signal,Price,RSI
0,SPY,BUY,734.42,61.52
22,ABBV,BUY,233.78,60.45
23,MRK,BUY,117.82,57.17
48,XLF,BUY,53.91,59.86
25,GE,BUY,355.92,64.83
27,DE,BUY,597.41,58.64
28,HON,BUY,224.67,52.58
33,SBUX,BUY,100.00,53.77
38,MU,BUY,1077.20,77.06
39,QCOM,BUY,199.25,55.31


In [101]:
portfolio = pd.DataFrame()

for ticker, df in results.items():
    portfolio[ticker] = df["strategy_return"]

portfolio = portfolio.fillna(0)

# Equal-weight portfolio
portfolio["strategy"] = portfolio.mean(axis=1)

# Buy-and-hold benchmark
market = pd.DataFrame()

for ticker, df in results.items():
    market[ticker] = df["weekly_return"]

market = market.fillna(0)
market["market"] = market.mean(axis=1)

In [102]:
def calc_metrics(series, weeks):

    subset = series.tail(weeks)

    total_return = (1 + subset).prod() - 1

    annualized_return = (
        (1 + total_return)**(52/weeks) - 1
    )

    volatility = subset.std() * np.sqrt(52)

    sharpe = (
        annualized_return / volatility
        if volatility > 0
        else np.nan
    )

    wealth = (1 + subset).cumprod()
    drawdown = wealth / wealth.cummax() - 1

    return {
        "Return %": round(total_return * 100, 2),
        "Annualized %": round(annualized_return * 100, 2),
        "Volatility %": round(volatility * 100, 2),
        "Sharpe": round(sharpe, 2),
        "Max DD %": round(drawdown.min() * 100, 2)
    }


periods = {
    "3M": 13,
    "6M": 26,
    "9M": 39,
    "12M": 52,
    "24M": 104
}

rows = []

for label, weeks in periods.items():

    strat = calc_metrics(
        portfolio["strategy"],
        weeks
    )

    bench = calc_metrics(
        market["market"],
        weeks
    )

    rows.append({
        "Period": label,

        "Strategy Return %": strat["Return %"],
        "Market Return %": bench["Return %"],

        "Strategy Sharpe": strat["Sharpe"],
        "Market Sharpe": bench["Sharpe"],

        "Strategy MaxDD %": strat["Max DD %"],
        "Market MaxDD %": bench["Max DD %"]
    })

summary = pd.DataFrame(rows)

summary

,Period,Strategy Return %,Market Return %,Strategy Sharpe,Market Sharpe,Strategy MaxDD %,Market MaxDD %
0,3M,12.56,22.51,7.14,9.32,-1.48,-1.91
1,6M,14.54,22.02,3.69,3.75,-3.64,-6.59
2,9M,21.56,30.89,3.24,3.27,-3.64,-6.59
3,12M,32.52,45.92,3.69,3.69,-3.64,-6.59
4,24M,36.71,69.10,1.72,1.76,-9.21,-17.28


In [103]:
for ticker, df in results.items():

    invested_pct = (
        df["position"].mean() * 100
    )

    print(
        f"{ticker}: {invested_pct:.1f}% invested"
    )

SPY: 73.4% invested
QQQ: 72.5% invested
DIA: 72.2% invested
IWM: 58.9% invested
AAPL: 63.9% invested
MSFT: 64.6% invested
NVDA: 64.3% invested
GOOGL: 63.7% invested
AMZN: 59.4% invested
META: 57.3% invested
TSLA: 50.3% invested
AVGO: 73.8% invested
ORCL: 65.9% invested
AMD: 58.0% invested
JPM: 66.6% invested
BAC: 60.7% invested
GS: 61.6% invested
MS: 62.8% invested
BLK: 59.8% invested
LLY: 68.8% invested
UNH: 61.4% invested
JNJ: 61.2% invested
ABBV: 59.4% invested
MRK: 62.5% invested
CAT: 58.2% invested
GE: 55.5% invested
RTX: 67.5% invested
DE: 58.5% invested
HON: 57.1% invested
COST: 69.8% invested
WMT: 71.6% invested
HD: 57.8% invested
MCD: 67.7% invested
SBUX: 52.4% invested
XOM: 54.9% invested
CVX: 54.4% invested
COP: 49.2% invested
SLB: 38.1% invested
MU: 53.3% invested
QCOM: 51.9% invested
TXN: 60.5% invested
INTC: 43.6% invested
ASML: 62.5% invested
VTI: 72.0% invested
VOO: 73.6% invested
SCHD: 70.0% invested
VXUS: 63.7% invested
XLK: 71.6% invested
XLF: 65.0% invested
XLE: 56.